<div>
    <img src="https://storage.googleapis.com/kaggle-datasets-images/19/19/default-backgrounds/dataset-cover.jpg"/>
</div>

<h3 class="list-group-item list-group-item-action active" data-toggle="list" style="color:white; background:#a185cf; border:1px dashed green;" role="tab" aria-controls="home"><center>Prepare Data</center></h3>

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from functools import reduce
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv('/kaggle/input/iris/Iris.csv')
df.head()

In [ ]:
species = dict(zip(list(df['Species'].unique()), ([0, 1, 2])))
print("Species:",species)

In [ ]:
categories = { v:k for (k,v) in species.items() }
print("Categories:", categories)

In [ ]:
# Replace species by one hot encodings
df['Species'].replace(species, inplace=True)
df['Species'].unique()

# Training DataFrame
iris = df[['PetalLengthCm', 'PetalWidthCm', 'Species']]
iris.head()

In [ ]:
d = 2

features = iris[['PetalLengthCm', 'PetalWidthCm']].values
labels = iris['Species'].values

X = torch.from_numpy(features.astype(np.float32))
y = torch.from_numpy(labels)

<h3 class="list-group-item list-group-item-action active" data-toggle="list" style="color:white; background:#a185cf; border:1px dashed green;" role="tab" aria-controls="home"><center>Implementation</center></h3>

In [ ]:
def torch_kron_prod(a, b):
    res = torch.einsum('ij,ik->ijk', [a, b])
    res = torch.reshape(res, [-1, np.prod(res.shape[1:])])
    return res

In [ ]:
def torch_bin(x, cut_points, temperature=0.1):
    # x is a N-by-1 matrix (column vector)
    # cut_points is a D-dim vector (D is the number of cut-points)
    # this function produces a N-by-(D+1) matrix, each row has only one element being one and the rest are all zeros
    D = cut_points.shape[0]
    W = torch.reshape(torch.linspace(1.0, D + 1.0, D + 1), [1, -1])
    cut_points, _ = torch.sort(cut_points)  # make sure cut_points is monotonically increasing
    b = torch.cumsum(torch.cat([torch.zeros([1]), -cut_points], 0),0)
    h = torch.matmul(x, W) + b
    res = torch.exp(h-torch.max(h))
    res = res/torch.sum(res, dim=-1, keepdim=True)
    return h

In [ ]:
def nn_decision_tree(x, cut_points_list, leaf_score, temperature=0.1):
    # cut_points_list contains the cut_points for each dimension of feature
    leaf = reduce(torch_kron_prod,
                  map(lambda z: torch_bin(x[:, z[0]:z[0] + 1], z[1], temperature), enumerate(cut_points_list)))
    return torch.matmul(leaf, leaf_score)

In [ ]:
# "Petal length" and "Petal width"
num_cut = [1, 1]
num_leaf = np.prod(np.array(num_cut) + 1)
num_class = 3

In [ ]:
cut_points_list = [torch.rand([i], requires_grad=True) for i in num_cut]
leaf_score = torch.rand([num_leaf, num_class], requires_grad=True)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cut_points_list + [leaf_score], lr=0.01)

In [ ]:
model = nn.Sequential(
          nn.Linear(2,16),
          nn.ReLU(),
          nn.Linear(16,2),
          nn.Sigmoid()
        )

<h3 class="list-group-item list-group-item-action active" data-toggle="list" style="color:white; background:#a185cf; border:1px dashed green;" role="tab" aria-controls="home"><center>Training</center></h3>

In [ ]:
for i in range(3000):
    optimizer.zero_grad()
    out = model(X)
    y_pred = nn_decision_tree(out, cut_points_list, leaf_score, temperature=0.1)
    loss = loss_function(y_pred, y)
    loss.backward()
    optimizer.step()
    if i % 200 == 0:
        print("i:{:4d} -- Loss:{:}".format(i, loss.detach().numpy()))
print('Error rate %.2f' % (1-np.mean(np.argmax(y_pred.detach().numpy(), axis=1)==y.detach().numpy())))

<h3 class="list-group-item list-group-item-action active" data-toggle="list" style="color:white; background:#a185cf; border:1px dashed green;" role="tab" aria-controls="home"><center>Prediction</center></h3>

In [ ]:
def predict(petal_length, petal_width):
    X = torch.tensor([[petal_length, petal_width]]).type(torch.float32)
    out = model(X)
    y_pred = nn_decision_tree(out, cut_points_list, leaf_score, temperature=0.1)
    return categories[torch.max(y_pred, axis=1)[1].item()]

In [ ]:
# predict using parameters

petal_length = 2.0
petal_width = 4.0

category = predict(petal_length, petal_width)

print("Category:", category)